<a href="https://colab.research.google.com/github/lspnzz/granted-search-engine/blob/main/evals/notebooks/run_search_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Granted Search Evals**

The most important thing for us is that our service retrieves all possible grants that could be relevant for each pitch. We're ok with getting less relevant grants, as long as we don't miss relevant ones.

We only care about document-level retrieval, we don't care about specific sections of the document.

First we need to make sure our app does what it's supposed to do, then we'll figure out how to make it more precise and efficient.

### **Primary metric**
- **Recall:** measures how many of the relevant documents were successfully retrieved. It focuses on not missing important results. Higher recall means fewer relevant documents were left out.

### **Secondary metrics**

We will evaluate these metrics at a later time, once we've improved the effectiveness of the system.

- **context relevance:** make sure all the retrieved chunks make sense for the given input;
- **context precision:** more relevant chunks are ranked higher than others;


## **Run searches for the Golden Dataset**

Configure the run:

In [20]:
from datetime import date
today = date.today()
date_str = today.strftime("%Y-%m-%d")

MODEL_NAME = input("Enter the model name: ")
MODEL_DIMENSIONS = input("Enter the model dimensions: ")
METRIC = input("Enter the distance metric: ")
CHUNK_SIZE = input("Enter the chunk size: ")
CHUNK_OVERLAP = input("Enter the chunk overlap: ")
k = int(input("Enter the number of results to retrieve: "))

RUN_ID = f"{date_str}_model-{MODEL_NAME}_dimensions-{MODEL_DIMENSIONS}_metric-{METRIC}_chunk-size-{CHUNK_SIZE}_chunk-overlap-{CHUNK_OVERLAP}_top-{k}_recall.csv"

SEARCH_URL = input("Enter the search function url: ")
PINECONE_INDEX_HOST = input("Enter the pinecone index host: ")
PINECONE_NAMESPACE = input("Enter the pinecone namespace: ")


Enter the model name: openai-embedding-3-large
Enter the model dimensions: 3072
Enter the distance metric: cosine
Enter the chunk size: 2000
Enter the chunk overlap: 200
Enter the number of results to retrieve: 10
Enter the search function url: https://search-grants-embedding-large-497910573020.europe-west1.run.app
Enter the pinecone index host: https://eval-eu-grants-openai-embedding-3-large-cg1nf1q.svc.aped-4627-b74a.pinecone.io
Enter the pinecone namespace: eu-grant-chunks


Create the run dataframe:

In [21]:
from google.colab import userdata
from tqdm import tqdm
import pandas as pd
import requests

# (LS): Load Golden dataset from Google Drive.
file_id = "1B0QZQf6xDjj01ViGjYY33J_D_TVB-LOf"  # the part after /d/ and before /view
golden_url = f"https://drive.google.com/uc?export=download&id={file_id}"
golden_df = pd.read_csv(golden_url, sep=";")

train_size = 0.7
train_df = golden_df.sample(frac=train_size, random_state=42)
results = []

k = 10

for idx, row in tqdm(train_df.iterrows(), total=len(train_df)):
    pitch_id = row["pitch_id"]
    pitch_text = row["pitch"]

    try:
        payload = {
            "pitch": pitch_text,
            "top_k": k,
            "pinecone_index_host": PINECONE_INDEX_HOST,
            "pinecone_namespace":  PINECONE_NAMESPACE,
        }

        HEADERS = {"Content-Type": "application/json"}
        response = requests.post(SEARCH_URL, headers=HEADERS, json=payload)
        data = response.json()
        matched_grants = data.get("grants", [])

        results.append({
            "pitch_id": pitch_id,
            "matched_grant_chunks_ids": [grant["id"] for grant in matched_grants],
        })

    except Exception as e:
        print(f"Error processing pitch {pitch_id}: {e}")

run_df = pd.DataFrame(results)
run_df.to_csv(f"{RUN_ID}.csv")

100%|██████████| 53/53 [01:19<00:00,  1.51s/it]


## **Compute Recall@k**

Process retrieved grant chunks to extract the grant ID:

In [22]:
def get_grant_id_from_chunk(chunk_id):
    if isinstance(chunk_id, str) and '-' in chunk_id:
        return chunk_id.rsplit('-', 1)[0]
    return chunk_id

run_df["matched_grant_ids"] = run_df["matched_grant_chunks_ids"].apply(lambda x: [get_grant_id_from_chunk(chunk_id) for chunk_id in x])

Merge the run results with the expected answers from the Golden dataset:

In [23]:
merged_df = pd.merge(train_df, run_df, on="pitch_id", how="inner")

We'll temporarily exclude cases where no relevant chunks should have been returned. We'll evaluate these cases separately once the system is updated to handle them ("False alarm rate" on the negative set).

> **TODO(LS):** Update search engine to handle "no relevant grants" case.

In [27]:
merged_df = merged_df[merged_df["matching_grant_ids"].notna()]

Compute the recall:

In [28]:
def compute_recall(row):
    true_grants = set(row["matching_grant_ids"].split(", "))
    retrieved_grants = set(row["matched_grant_ids"])

    intersection = len(true_grants.intersection(retrieved_grants))
    return intersection / len(true_grants)

merged_df["recall"] = merged_df.apply(compute_recall, axis=1)
print(f"Mean Recall: {merged_df['recall'].mean():.4f}")

Mean Recall: 0.4209


/tmp/ipython-input-843232978.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df["recall"] = merged_df.apply(compute_recall, axis=1)


Save results (remember to safely store downloaded file):



In [29]:
from google.colab import files

results_filename = f"{RUN_ID}_recall.csv"
results_df = merged_df[['pitch_id', 'recall']]
results_df.to_csv(results_filename, index=False)
files.download(results_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>